In [1]:
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import numpy as np
import nibabel as nib
import shutil 

In [2]:
data_dir = Path("../../data_new")
old_data_dir = Path("../../data")
tmp_data_dir = Path("../../data_tmp")
data_dir.mkdir(exist_ok=True, parents=False)
(data_dir / "images").mkdir(exist_ok=True, parents=False)
(data_dir / "labels").mkdir(exist_ok=True, parents=False)
(data_dir / "splits").mkdir(exist_ok=True, parents=False)

In [3]:
ids, mask, bg, cso, age, sex, hyp, wmh, isl, lac, bld  = [], [], [], [], [], [], [], [], [], [], []

In [4]:
def to_bin(s):
    if s.lower() == "y":
        return 1
    elif s.lower() == "n":
        return 0
    else:
        return "N/A"

### MSS1

In [5]:
mss1_data_sheet = pd.read_excel("../../../../data/misc/data_MSS1.xlsx")
mss1_scores_sheet = pd.read_excel("../../../../MSS1_PVS_VRS.xlsx", sheet_name="Original_Data_FD")

mss1_data_root = Path("../../../../data/datasets/MSS1")

key = pd.read_excel("../../../../CVDkey_MVH.xlsx")
CVDscores = pd.read_excel("../../../../CVDclinScoresIXI.xlsx")

for row in tqdm(mss1_data_sheet.iterrows(), total=len(mss1_data_sheet)):
    if not (pd.notna(row[1]["T1w"]) and pd.notna(row[1]["T2w"]) and pd.notna(row[1]["FLAIR"])):
        continue
    
    id_ = row[1]["id"]

    try:
        CVDid = key[key["StudyID"] == id_]["ID"].values[0]
    except IndexError:
        CVDid = None

    T1w_path = mss1_data_root / row[1]["T1w"]
    T2w_path = mss1_data_root / row[1]["T2w"]
    FLAIR_path = mss1_data_root / row[1]["FLAIR"]

    bg_ = mss1_scores_sheet[mss1_scores_sheet["Unique identifier MSS"] == id_]["BasalGangliaOverall"].values[0]
    cso_ = mss1_scores_sheet[mss1_scores_sheet["Unique identifier MSS"] == id_]["CentrumOverall"].values[0]
    age_ = row[1]["age"]
    sex_ = row[1]["sex"]
    if CVDid is not None:
        hyp_ = CVDscores[CVDscores["ID"] == CVDid]["hypertension"].values[0]
    
    if not pd.isna(row[1]["WMH"]):
        WMH = nib.load(mss1_data_root / row[1]["WMH"])
        wmh_ = np.prod(WMH.header.get_zooms()) * np.sum(WMH.get_fdata())
    else:
        wmh_ = "N/A"

    if not pd.isna(row[1]["ISLtotal"]):
        ISL = nib.load(mss1_data_root / row[1]["ISLtotal"])
        isl_ = 1 if np.sum(ISL.get_fdata()) > 0 else 0
    else:
        isl_ = "N/A"

    lac_ = "N/A"
    if CVDid is not None:
        bld_ = 1 if CVDscores[CVDscores["ID"] == CVDid]["micrBld"].values[0] > 0 else 0
    else:
        bld_ = "N/A"
    
    if (old_data_dir / "images" / f"{id_}_T1w.nii.gz").exists():
        # shutil.copy(old_data_dir / "images" / f"{id_}_T1w.nii.gz", tmp_data_dir / "images" / f"{id_}_T1w.nii.gz")
        # shutil.copy(old_data_dir / "images" / f"{id_}_T2w.nii.gz", tmp_data_dir / "images" / f"{id_}_T2w.nii.gz")
        # shutil.copy(old_data_dir / "images" / f"{id_}_FLAIR.nii.gz", tmp_data_dir / "images" / f"{id_}_FLAIR.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_BG.nii.gz", tmp_data_dir / "labels" / f"{id_}_BG.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_CSO.nii.gz", tmp_data_dir / "labels" / f"{id_}_CSO.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_synthseg.nii.gz", tmp_data_dir / "labels" / f"{id_}_synthseg.nii.gz")
        shutil.copy(old_data_dir / "labels" / f"{id_}_BRAIN.nii.gz", tmp_data_dir / "labels" / f"{id_}_BRAIN.nii.gz")
    else:
        # shutil.copy(T1w_path, data_dir / "images" / f"{id_}_T1w.nii.gz")
        # shutil.copy(T2w_path, data_dir / "images" / f"{id_}_T2w.nii.gz")
        # shutil.copy(FLAIR_path, data_dir / "images" / f"{id_}_FLAIR.nii.gz")
        pass
    
    ids.append(id_)
    mask.append("n")
    bg.append(bg_ if not pd.isna(bg_) else "N/A")
    cso.append(cso_ if not pd.isna(cso_) else "N/A")
    age.append(age_ if not pd.isna(age_) else "N/A")
    sex.append(sex_ if not pd.isna(sex_) else "N/A")
    hyp.append(hyp_ if not pd.isna(hyp_) else "N/A")
    wmh.append(wmh_ if not pd.isna(wmh_) else "N/A")
    isl.append(isl_ if not pd.isna(isl_) else "N/A")
    lac.append(lac_ if not pd.isna(lac_) else "N/A")
    bld.append(bld_ if not pd.isna(bld_) else "N/A")


  0%|          | 0/96 [00:00<?, ?it/s]

### MSS2

In [6]:
mss2_data_sheet = pd.read_excel("../../../../data/misc/data_MSS2.xlsx")
mss2_scores_sheet = pd.read_excel("../../../../AllData_MATLab_v2.xlsx", sheet_name="All_data_from_Cesca")

mss2_data_root = Path("../../../../data/datasets/MSS2")

key = pd.read_excel("../../../../CVDkey_MVH.xlsx")
CVDscores = pd.read_excel("../../../../CVDclinScoresIXI.xlsx")

for row in tqdm(mss2_data_sheet.iterrows(), total=len(mss2_data_sheet)):
    if not (pd.notna(row[1]["T1w"]) and pd.notna(row[1]["T2w"]) and pd.notna(row[1]["FLAIR"])):
        continue
    
    id_ = row[1]["id"]

    try:
        CVDid = key[key["studySpecID"] == id_]["ID"].values[0]
    except IndexError:
        CVDid = None

    T1w_path = mss2_data_root / row[1]["T1w"]
    T2w_path = mss2_data_root / row[1]["T2w"]
    FLAIR_path = mss2_data_root / row[1]["FLAIR"]
    if not pd.isna(row[1]["PVS"]):
        PVS_path = mss2_data_root/ row[1]["PVS"]

    try:
        bg_ = mss2_scores_sheet[mss2_scores_sheet["MssRegid"] == id_]["f03_EPVL_BASALOverall"].values[0]
    except IndexError:
        bg_ = "N/A"
    try:
        cso_ = mss2_scores_sheet[mss2_scores_sheet["MssRegid"] == id_]["f03_EPVL_CENTOverall"].values[0]
    except IndexError:
        cso_ = "N/A"
    
    try:
        sex_ = mss2_scores_sheet[mss2_scores_sheet["MssRegid"] == id_]["f01_Sex"].values[0]
    except IndexError:
        sex_ = "N/A"

    try:
        age_ = mss2_scores_sheet[mss2_scores_sheet["MssRegid"] == id_]["age"].values[0]
    except IndexError:
        age_ = "N/A"

    if CVDid is not None:
        hyp_ = CVDscores[CVDscores["ID"] == CVDid]["hypertension"].values[0]
    
    if not pd.isna(row[1]["WMH"]):
        WMH = nib.load(mss2_data_root / row[1]["WMH"])
        wmh_ = np.prod(WMH.header.get_zooms()) * np.sum(WMH.get_fdata())
    else:
        wmh_ = "N/A"

    if not pd.isna(row[1]["ISLtotal"]):
        ISL = nib.load(mss2_data_root / row[1]["ISLtotal"])
        isl_ = 1 if np.sum(ISL.get_fdata()) > 0 else 0
    else:
        isl_ = "N/A"

    try:
        lac_ = 1 if mss2_scores_sheet[mss2_scores_sheet["MssRegid"] == id_]["lacs"].values[0] > 0 else 0
    except IndexError:
        lac_ = "N/A"
    try:
        bld_ = mss2_scores_sheet[mss2_scores_sheet["MssRegid"] == id_]["svdmb"].values[0]
    except IndexError:
        bld_ = "N/A"
    

    if (old_data_dir / "images" / f"{id_}_T1w.nii.gz").exists():
        # shutil.copy(old_data_dir / "images" / f"{id_}_T1w.nii.gz", tmp_data_dir / "images" / f"{id_}_T1w.nii.gz")
        # shutil.copy(old_data_dir / "images" / f"{id_}_T2w.nii.gz", tmp_data_dir / "images" / f"{id_}_T2w.nii.gz")
        # shutil.copy(old_data_dir / "images" / f"{id_}_FLAIR.nii.gz", tmp_data_dir / "images" / f"{id_}_FLAIR.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_BG.nii.gz", tmp_data_dir / "labels" / f"{id_}_BG.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_CSO.nii.gz", tmp_data_dir / "labels" / f"{id_}_CSO.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_synthseg.nii.gz", tmp_data_dir / "labels" / f"{id_}_synthseg.nii.gz")
        shutil.copy(old_data_dir / "labels" / f"{id_}_BRAIN.nii.gz", tmp_data_dir / "labels" / f"{id_}_BRAIN.nii.gz")
        if not pd.isna(row[1]["PVS"]):
            # shutil.copy(old_data_dir / "labels" / f"{id_}_PVS.nii.gz", tmp_data_dir / "labels" / f"{id_}_PVS.nii.gz")
            mask.append("y")
        else:
            mask.append("n")
    else:
        # shutil.copy(T1w_path, data_dir / "images" / f"{id_}_T1w.nii.gz")
        # shutil.copy(T2w_path, data_dir / "images" / f"{id_}_T2w.nii.gz")
        # shutil.copy(FLAIR_path, data_dir / "images" / f"{id_}_FLAIR.nii.gz")
        if not pd.isna(row[1]["PVS"]):
            # shutil.copy(PVS_path, data_dir / "labels" / f"{id_}_PVS.nii.gz")
            mask.append("y")
        else:
            mask.append("n")
    
    ids.append(id_)
    bg.append(bg_ if not pd.isna(bg_) else "N/A")
    cso.append(cso_ if not pd.isna(cso_) else "N/A")
    age.append(age_ if not pd.isna(age_) else "N/A")
    sex.append(sex_ if not pd.isna(sex_) else "N/A")
    hyp.append(hyp_ if not pd.isna(hyp_) else "N/A")
    wmh.append(wmh_ if not pd.isna(wmh_) else "N/A")
    isl.append(isl_ if not pd.isna(isl_) else "N/A")
    lac.append(lac_ if not pd.isna(lac_) else "N/A")
    bld.append(bld_ if not pd.isna(bld_) else "N/A")


  0%|          | 0/261 [00:00<?, ?it/s]

### MSS3

In [7]:
mss3_data_sheet = pd.read_excel("../../../../data/misc/data_MSS3.xlsx")
mss3_scores_sheet = pd.read_excel("../../../../MVH_JPhitidis_Pt_20230512.xlsx")

mss3_data_root = Path("../../../../data/datasets/MSS3")

for row in tqdm(mss3_data_sheet.iterrows(), total=len(mss3_data_sheet)):
    if not (pd.notna(row[1]["T1w"]) and pd.notna(row[1]["T2w"]) and pd.notna(row[1]["FLAIR"])):
        continue
    
    id_ = row[1]["id"]

    T1w_path = mss3_data_root / row[1]["T1w"]
    T2w_path = mss3_data_root / row[1]["T2w"]
    FLAIR_path = mss3_data_root / row[1]["FLAIR"]
    if not pd.isna(row[1]["PVS"]):
        PVS_path = mss3_data_root/ row[1]["PVS"]

    bg_ = mss3_scores_sheet[mss3_scores_sheet["MSS3_Study_Number"] == id_]["V1_PSBGOverall"].values[0]
    cso_ = mss3_scores_sheet[mss3_scores_sheet["MSS3_Study_Number"] == id_]["V1_PSCSOverall"].values[0]
    age_ = row[1]["age"]
    sex_ = row[1]["sex"]
    hyp_ = mss3_scores_sheet[mss3_scores_sheet["MSS3_Study_Number"] == id_]["hypertensionDiagnosis"].values[0]
    
    if not pd.isna(row[1]["WMH"]):
        WMH = nib.load(mss3_data_root / row[1]["WMH"])
        wmh_ = np.prod(WMH.header.get_zooms()) * np.sum(WMH.get_fdata())
    else:
        wmh_ = "N/A"

    if not pd.isna(row[1]["ISLtotal"]):
        ISL = nib.load(mss3_data_root / row[1]["ISLtotal"])
        isl_ = 1 if np.sum(ISL.get_fdata()) > 0 else 0
    else:
        isl_ = "N/A"

    lac_ = to_bin(mss3_scores_sheet[mss3_scores_sheet["MSS3_Study_Number"] == id_]["V1_LacuneVisible"].values[0])
    bld_ = to_bin(mss3_scores_sheet[mss3_scores_sheet["MSS3_Study_Number"] == id_]["V1_Micro_Present"].values[0])
    
    if (old_data_dir / "images" / f"{id_}_T1w.nii.gz").exists():
        # shutil.copy(old_data_dir / "images" / f"{id_}_T1w.nii.gz", tmp_data_dir / "images" / f"{id_}_T1w.nii.gz")
        # shutil.copy(old_data_dir / "images" / f"{id_}_T2w.nii.gz", tmp_data_dir / "images" / f"{id_}_T2w.nii.gz")
        # shutil.copy(old_data_dir / "images" / f"{id_}_FLAIR.nii.gz", tmp_data_dir / "images" / f"{id_}_FLAIR.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_BG.nii.gz", tmp_data_dir / "labels" / f"{id_}_BG.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_CSO.nii.gz", tmp_data_dir / "labels" / f"{id_}_CSO.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_synthseg.nii.gz", tmp_data_dir / "labels" / f"{id_}_synthseg.nii.gz")
        shutil.copy(old_data_dir / "labels" / f"{id_}_BRAIN.nii.gz", tmp_data_dir / "labels" / f"{id_}_BRAIN.nii.gz")
        if not pd.isna(row[1]["PVS"]):
            # shutil.copy(old_data_dir / "labels" / f"{id_}_PVS.nii.gz", tmp_data_dir / "labels" / f"{id_}_PVS.nii.gz")
            mask.append("y")
        else:
            mask.append("n")
    else:
        # shutil.copy(T1w_path, data_dir / "images" / f"{id_}_T1w.nii.gz")
        # shutil.copy(T2w_path, data_dir / "images" / f"{id_}_T2w.nii.gz")
        # shutil.copy(FLAIR_path, data_dir / "images" / f"{id_}_FLAIR.nii.gz")
        if not pd.isna(row[1]["PVS"]):
            # shutil.copy(PVS_path, data_dir / "labels" / f"{id_}_PVS.nii.gz")
            mask.append("y")
        else:
            mask.append("n")
    
    ids.append(id_)
    bg.append(bg_ if not pd.isna(bg_) else "N/A")
    cso.append(cso_ if not pd.isna(cso_) else "N/A")
    age.append(age_ if not pd.isna(age_) else "N/A")
    sex.append(sex_ if not pd.isna(sex_) else "N/A")
    hyp.append(hyp_ if not pd.isna(hyp_) else "N/A")
    wmh.append(wmh_ if not pd.isna(wmh_) else "N/A")
    isl.append(isl_ if not pd.isna(isl_) else "N/A")
    lac.append(lac_ if not pd.isna(lac_) else "N/A")
    bld.append(bld_ if not pd.isna(bld_) else "N/A")

  0%|          | 0/229 [00:00<?, ?it/s]

### LBC1936

In [8]:
lbc1936_data_sheet = pd.read_excel("../../../../data/misc/data_LBC1936.xlsx")
lbc1936_scores_sheet = pd.read_excel("../../../../../LBC1936/LBC_1936_Inflammation_data/LBC1936_InflammationAndBrain_AndCarotidPerfBp_JMW_merged_used_02Nov2012.xls")

lbc1936_data_root = Path("../../../../data/datasets/LBC1936")

key = pd.read_excel("../../../../CVDkey_MVH.xlsx")
CVDscores = pd.read_excel("../../../../CVDclinScoresIXI.xlsx")

for row in tqdm(lbc1936_data_sheet.iterrows(), total=len(lbc1936_data_sheet)):
    if not (pd.notna(row[1]["T1w"]) and pd.notna(row[1]["T2w"]) and pd.notna(row[1]["FLAIR"])):
        continue
    
    id_ = row[1]["id"]

    try:
        CVDid = key[key["studySpecID"] == id_]["ID"].values[0]
    except IndexError:
        CVDid = None

    T1w_path = lbc1936_data_root / row[1]["T1w"]
    T2w_path = lbc1936_data_root / row[1]["T2w"]
    FLAIR_path = lbc1936_data_root / row[1]["FLAIR"]
    if not pd.isna(row[1]["PVS"]):
        PVS_path = lbc1936_data_root/ row[1]["PVS"]

    try:
        bg_ = lbc1936_scores_sheet[lbc1936_scores_sheet["lbc36no"] == id_]["enl_peri_space_bgo_w2"].values[0]
    except IndexError:
        bg_ = "N/A"
    try:
        cso_ = lbc1936_scores_sheet[lbc1936_scores_sheet["lbc36no"] == id_]["enl_peri_space_coo_w2"].values[0]
    except IndexError:
        cso_ = "N/A"
    
    sex_ = row[1]["sex"]

    try:
        age_ = lbc1936_scores_sheet[lbc1936_scores_sheet["lbc36no"] == id_]["ageMRI_w2"].values[0] / 365
    except IndexError:
        age_ = "N/A"

    if CVDid is not None:
        hyp_ = CVDscores[CVDscores["ID"] == CVDid]["hypertension"].values[0]
    
    if not pd.isna(row[1]["WMH"]):
        WMH = nib.load(lbc1936_data_root / row[1]["WMH"])
        wmh_ = np.prod(WMH.header.get_zooms()) * np.sum(WMH.get_fdata())
    else:
        wmh_ = "N/A"

    if not pd.isna(row[1]["ISLtotal"]):
        ISL = nib.load(lbc1936_data_root / row[1]["ISLtotal"])
        isl_ = 1 if np.sum(ISL.get_fdata()) > 0 else 0
    else:
        isl_ = "N/A"

    lac_ = "N/A"

    if CVDid is not None:
        bld_ = 1 if CVDscores[CVDscores["ID"] == CVDid]["micrBld"].values[0] > 0 else 0
    else:
        bld_ = "N/A"

    if (old_data_dir / "images" / f"{id_}_T1w.nii.gz").exists():
        # shutil.copy(old_data_dir / "images" / f"{id_}_T1w.nii.gz", tmp_data_dir / "images" / f"{id_}_T1w.nii.gz")
        # shutil.copy(old_data_dir / "images" / f"{id_}_T2w.nii.gz", tmp_data_dir / "images" / f"{id_}_T2w.nii.gz")
        # shutil.copy(old_data_dir / "images" / f"{id_}_FLAIR.nii.gz", tmp_data_dir / "images" / f"{id_}_FLAIR.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_BG.nii.gz", tmp_data_dir / "labels" / f"{id_}_BG.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_CSO.nii.gz", tmp_data_dir / "labels" / f"{id_}_CSO.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_synthseg.nii.gz", tmp_data_dir / "labels" / f"{id_}_synthseg.nii.gz")
        shutil.copy(old_data_dir / "labels" / f"{id_}_BRAIN.nii.gz", tmp_data_dir / "labels" / f"{id_}_BRAIN.nii.gz")
        if not pd.isna(row[1]["PVS"]):
            # shutil.copy(old_data_dir / "labels" / f"{id_}_PVS.nii.gz", tmp_data_dir / "labels" / f"{id_}_PVS.nii.gz")
            mask.append("y")
        else:
            mask.append("n")
    else:
        # shutil.copy(T1w_path, data_dir / "images" / f"{id_}_T1w.nii.gz")
        # shutil.copy(T2w_path, data_dir / "images" / f"{id_}_T2w.nii.gz")
        # shutil.copy(FLAIR_path, data_dir / "images" / f"{id_}_FLAIR.nii.gz")
        if not pd.isna(row[1]["PVS"]):
            # shutil.copy(PVS_path, data_dir / "labels" / f"{id_}_PVS.nii.gz")
            mask.append("y")
        else:
            mask.append("n")
    
    ids.append(id_)
    bg.append(bg_ if not pd.isna(bg_) else "N/A")
    cso.append(cso_ if not pd.isna(cso_) else "N/A")
    age.append(age_ if not pd.isna(age_) else "N/A")
    sex.append(sex_ if not pd.isna(sex_) else "N/A")
    hyp.append(hyp_ if not pd.isna(hyp_) else "N/A")
    wmh.append(wmh_ if not pd.isna(wmh_) else "N/A")
    isl.append(isl_ if not pd.isna(isl_) else "N/A")
    lac.append(lac_ if not pd.isna(lac_) else "N/A")
    bld.append(bld_ if not pd.isna(bld_) else "N/A")



  0%|          | 0/696 [00:00<?, ?it/s]

### VALDO 1

In [5]:
valdo1_data_sheet = pd.read_excel("../../../../data/misc/data_VALDO1.xlsx")

valdo1_data_root = Path("../../../../data/datasets/VALDO1")

for row in tqdm(valdo1_data_sheet.iterrows(), total=len(valdo1_data_sheet)):
    if not (pd.notna(row[1]["T1w"]) and pd.notna(row[1]["T2w"]) and pd.notna(row[1]["FLAIR"]) and pd.notna(row[1]["PVS"])):
        continue
    
    id_ = "VALDO1_" + str(row[1]["id"]).zfill(3)
    T1w_path = valdo1_data_root / row[1]["T1w"]
    T2w_path = valdo1_data_root / row[1]["T2w"]
    FLAIR_path = valdo1_data_root / row[1]["FLAIR"]
    PVS_path = valdo1_data_root/ row[1]["PVS"]
    bg_ = "N/A"
    cso_ = "N/A"
    age_ = "N/A"
    sex_ = "N/A"
    hyp_ = "N/A"
    wmh_ = "N/A"
    isl_ = "N/A"
    lac_ = "N/A"
    bld_ = "N/A"

    if (old_data_dir / "images" / f"{id_}_T1w.nii.gz").exists():
        # shutil.copy(old_data_dir / "images" / f"{id_}_T1w.nii.gz", tmp_data_dir / "images" / f"{id_}_T1w.nii.gz")
        # shutil.copy(old_data_dir / "images" / f"{id_}_T2w.nii.gz", tmp_data_dir / "images" / f"{id_}_T2w.nii.gz")
        # shutil.copy(old_data_dir / "images" / f"{id_}_FLAIR.nii.gz", tmp_data_dir / "images" / f"{id_}_FLAIR.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_PVS.nii.gz", tmp_data_dir / "labels" / f"{id_}_PVS.nii.gz")
        shutil.copy(old_data_dir / "labels" / f"{id_}_BG.nii.gz", tmp_data_dir / "labels" / f"{id_}_BG.nii.gz")
        shutil.copy(old_data_dir / "labels" / f"{id_}_CSO.nii.gz", tmp_data_dir / "labels" / f"{id_}_CSO.nii.gz")
        shutil.copy(old_data_dir / "labels" / f"{id_}_synthseg.nii.gz", tmp_data_dir / "labels" / f"{id_}_synthseg.nii.gz")
        # shutil.copy(old_data_dir / "labels" / f"{id_}_BRAIN.nii.gz", tmp_data_dir / "labels" / f"{id_}_BRAIN.nii.gz")
    else:
        # shutil.copy(T1w_path, data_dir / "images" / f"{id_}_T1w.nii.gz")
        # shutil.copy(T2w_path, data_dir / "images" / f"{id_}_T2w.nii.gz")
        # shutil.copy(FLAIR_path, data_dir / "images" / f"{id_}_FLAIR.nii.gz")
        # shutil.copy(PVS_path, data_dir / "labels" / f"{id_}_PVS.nii.gz")
        pass
    
    ids.append(id_)
    mask.append("y")
    bg.append(bg_ if not pd.isna(bg_) else "N/A")
    cso.append(cso_ if not pd.isna(cso_) else "N/A")
    age.append(age_ if not pd.isna(age_) else "N/A")
    sex.append(sex_ if not pd.isna(sex_) else "N/A")
    hyp.append(hyp_ if not pd.isna(hyp_) else "N/A")
    wmh.append(wmh_ if not pd.isna(wmh_) else "N/A")
    isl.append(isl_ if not pd.isna(isl_) else "N/A")
    lac.append(lac_ if not pd.isna(lac_) else "N/A")
    bld.append(bld_ if not pd.isna(bld_) else "N/A")

  0%|          | 0/12 [00:00<?, ?it/s]

In [10]:
df = pd.DataFrame({
    "id": ids,
    "mask": mask,
    "bg": bg,
    "cso": cso,
    "age": age,
    "sex": sex,
    "hyp": hyp,
    "wmh": wmh,
    "isl": isl,
    "lac": lac,
    "bld": bld,
    })

# Derived measures
df["bg_derivative"] = df["bg"].replace({0: 1, 1: 1, 2: 2, 3: 3, 4: 3})
df["cso_derivative"] = df["cso"].replace({0: 1, 1: 1, 2: 2, 3: 3, 4: 3})

# df.to_csv(data_dir / "metadata.csv", index=False)